# Anonymization Techniques: Applying Anonymization and Pseudonymization Methods

## 📚 Learning Objectives

By completing this notebook, you will:
- Apply anonymization techniques
- Apply pseudonymization methods
- Protect personal data
- Understand k-anonymity
- Implement privacy-preserving methods

## 🔗 Prerequisites

- ✅ Understanding of privacy
- ✅ Understanding of data protection
- ✅ Python knowledge

---

This notebook covers practical activities from **Course 06, Unit 3**:
- Anonymization Techniques: Applying anonymization and pseudonymization methods

---

## Introduction

**Anonymization and pseudonymization** protect personal data by removing or replacing identifying information, enabling data use while preserving privacy.

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# Concept map: the two ways to de-identify data and the techniques behind them.
# Why first: the hands-on cell below applies these terms - read them here once.

import pandas as pd
import numpy as np
import hashlib

print("✅ Libraries imported!")
print("\nAnonymization and Pseudonymization")
print("=" * 60)

# Anonymization is IRREVERSIBLE: once done properly, no one can link the data
# back to a person - which is why GDPR no longer applies to truly anonymous data.
print("\nAnonymization:")
print("  - Remove identifiers")
print("  - Generalize data")
print("  - Suppress values")
print("  - k-anonymity")

# Pseudonymization is REVERSIBLE by whoever holds the mapping/salt -
# the data is still personal data under GDPR.
print("\nPseudonymization:")
print("  - Replace identifiers")
print("  - Reversible mapping")
print("  - Hash functions")
print("  - Tokenization")

print("\nTechniques:")
print("  - Generalization")
print("  - Suppression")
print("  - Perturbation")
print("  - Data masking")

print("\n✅ Anonymization concepts understood!")

✅ Libraries imported!

Anonymization and Pseudonymization

Anonymization:
  - Remove identifiers
  - Generalize data
  - Suppress values
  - k-anonymity

Pseudonymization:
  - Replace identifiers
  - Reversible mapping
  - Hash functions
  - Tokenization

Techniques:
  - Generalization
  - Suppression
  - Perturbation
  - Data masking

✅ Anonymization concepts understood!


## Hands-on: Anonymization, Pseudonymization, and k-Anonymity

**k-anonymity**: a table is *k*-anonymous if every combination of **quasi-identifiers**
(attributes that could identify someone when combined - like age + zipcode) appears in at
least *k* rows. If a combination is unique (k = 1), that person can be singled out.

Below we apply anonymization and pseudonymization to a real table, *measure* its k, and
raise k by **generalizing** the quasi-identifiers - exactly what Exercise 2 (Task 1) will
ask you to implement yourself.

In [2]:
# Practice: pseudonymize, then measure and improve k-anonymity

df = pd.DataFrame({
    'name':    ['Sara', 'Omar', 'Lina', 'Adam', 'Maya', 'Ziad', 'Nora', 'Karim'],
    'age':     [34, 29, 41, 52, 38, 34, 29, 41],
    'zipcode': ['11564', '11321', '11564', '12211', '11321', '11565', '11322', '11563'],
    'condition': ['A', 'B', 'A', 'C', 'B', 'A', 'C', 'B'],
})
print("Original table:")
print(df.to_string(index=False))

# --- Pseudonymize the direct identifier ---
SALT = 'unit3-demo'
df_pseudo = df.copy()
df_pseudo['person_id'] = df_pseudo['name'].apply(
    lambda v: hashlib.sha256((SALT + v).encode()).hexdigest()[:10])
df_pseudo = df_pseudo.drop(columns=['name'])[['person_id', 'age', 'zipcode', 'condition']]
print("\nAfter pseudonymization (name -> salted hash):")
print(df_pseudo.to_string(index=False))

# --- Measure k-anonymity over the quasi-identifiers ---
def k_anonymity(table, quasi_identifiers):
    """Smallest group size over the quasi-identifier combinations that occur."""
    return int(table.groupby(quasi_identifiers, observed=True).size().min())

QI = ['age', 'zipcode']
k_before = k_anonymity(df_pseudo, QI)
print(f"\nk-anonymity over {QI}: k = {k_before}")
print("k = 1 means at least one person has a UNIQUE age+zipcode combination -")
print("an attacker who knows those two facts can single them out.")

# --- Step A: Generalize quasi-identifiers (coarser values -> bigger groups) ---
df_gen = df_pseudo.copy()
df_gen['age'] = pd.cut(df_gen['age'], bins=[0, 40, 120],
                       labels=['<=40', '41+']).astype(str)
df_gen['zipcode'] = df_gen['zipcode'].str[:3] + '**'
print("\nAfter generalization (age -> range, zipcode -> prefix):")
print(df_gen.to_string(index=False))
k_gen = k_anonymity(df_gen, QI)
print(f"k-anonymity after generalization: k = {k_gen}  (was {k_before})")

# --- Step B: Suppress rows still in groups smaller than the target k ---
TARGET_K = 2
group_sizes = df_gen.groupby(QI, observed=True)['person_id'].transform('size')
suppressed = df_gen[group_sizes < TARGET_K]
df_k = df_gen[group_sizes >= TARGET_K]
print(f"\nSuppressing {len(suppressed)} row(s) whose group is still smaller "
      f"than k={TARGET_K}:")
print(df_k.to_string(index=False))
k_after = k_anonymity(df_k, QI)
print(f"\nFinal k-anonymity: k = {k_after} (target was {TARGET_K})")
print("Generalize first, then suppress the stubborn outliers - the standard recipe.")
print("The cost: coarser values and dropped rows. Privacy vs utility, again.")
print("\n✅ You just did what Exercise 2 Task 1 asks: check k, generalize, re-check.")


Original table:
 name  age zipcode condition
 Sara   34   11564         A
 Omar   29   11321         B
 Lina   41   11564         A
 Adam   52   12211         C
 Maya   38   11321         B
 Ziad   34   11565         A
 Nora   29   11322         C
Karim   41   11563         B

After pseudonymization (name -> salted hash):
 person_id  age zipcode condition
9038e03208   34   11564         A
4d290e5d32   29   11321         B
d68bf688b3   41   11564         A
b73ca02d89   52   12211         C
5cf06d1d20   38   11321         B
df38f0791e   34   11565         A
ace96de0db   29   11322         C
5455297d58   41   11563         B

k-anonymity over ['age', 'zipcode']: k = 1
k = 1 means at least one person has a UNIQUE age+zipcode combination -
an attacker who knows those two facts can single them out.

After generalization (age -> range, zipcode -> prefix):
 person_id  age zipcode condition
9038e03208 <=40   115**         A
4d290e5d32 <=40   113**         B
d68bf688b3  41+   115**         A
b73

---

## ➡️ You've Finished Unit 3

With notebooks 06-07 done, you have now *practiced* the two core protection techniques
from Notebook 01 (encryption; anonymization/pseudonymization) and measured k-anonymity.

**Next**: the unit exercises (`exercises/exercise_01.ipynb`, `exercises/exercise_02.ipynb`),
then **Unit 4: Transparency and Accountability** (`unit4-transparency-accountability/`).

## 📚 References

1. Sweeney, L. (2002). *k-Anonymity: A Model for Protecting Privacy*. International Journal of Uncertainty, Fuzziness and Knowledge-Based Systems, 10(5).
2. Machanavajjhala, A., Kifer, D., Gehrke, J. & Venkitasubramaniam, M. (2007). *l-Diversity: Privacy Beyond k-Anonymity*. ACM Transactions on Knowledge Discovery from Data, 1(1).
3. Narayanan, A. & Shmatikov, V. (2007). *How To Break Anonymity of the Netflix Prize Dataset*. <https://arxiv.org/abs/cs/0610105>
4. European Parliament & Council (2016). *Regulation (EU) 2016/679 — General Data Protection Regulation (GDPR)*. <https://gdpr-info.eu/>